# PDF Stamp & Text Label Detection
Проект для детектирования печатей и текстовых меток в PDF документах

## 1. Установка библиотек

In [ ]:
!pip install opencv-python pillow pymupdf pandas tqdm matplotlib ultralytics -q

## 2. Импорты и подготовка

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.widgets import RectangleSelector

# Структура папок
INPUT_FOLDER = "input"
PAGES_FOLDER = "pages"
ANNOTATIONS_FOLDER = "annotations"
OUTPUT_FOLDER = "output"
DATASET_FOLDER = "dataset"  # Для YOLO

# Создаем папки если их нет
for folder in [INPUT_FOLDER, PAGES_FOLDER, ANNOTATIONS_FOLDER, OUTPUT_FOLDER, DATASET_FOLDER]:
    os.makedirs(folder, exist_ok=True)

print("✓ Папки созданы")

## 3. Конвертирование PDF в изображения

In [ ]:
def convert_pdf_to_images(pdf_path, output_dir=PAGES_FOLDER, dpi=300):
    """
    Конвертирует PDF в PNG изображения
    """
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    images = []
    
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img_path = f"{output_dir}/{Path(pdf_path).stem}_page_{i:03d}.png"
        pix.save(img_path)
        images.append(img_path)
    
    doc.close()
    return images

# Конвертируем все PDF из папки input
pdf_files = list(Path(INPUT_FOLDER).glob("*.pdf"))
print(f"Найдено PDF файлов: {len(pdf_files)}")

all_images = []
for pdf_file in tqdm(pdf_files, desc="Конвертирование PDF"):
    images = convert_pdf_to_images(str(pdf_file))
    all_images.extend(images)

print(f"✓ Всего изображений: {len(all_images)}")
print(f"Первое изображение: {all_images[0]}")

## 4. Интерактивная разметка печатей и меток

In [ ]:
class ImageAnnotator:
    def __init__(self, image_path, annotation_file):
        self.image_path = image_path
        self.annotation_file = annotation_file
        self.image = cv2.imread(image_path)
        self.image_rgb = cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB)
        self.bboxes = []
        self.load_annotations()
        
    def load_annotations(self):
        """Загружает существующие аннотации если есть"""
        if os.path.exists(self.annotation_file):
            with open(self.annotation_file, 'r') as f:
                data = json.load(f)
                self.bboxes = data.get('bboxes', [])
    
    def save_annotations(self):
        """Сохраняет аннотации в JSON"""
        data = {
            'image': self.image_path,
            'image_width': self.image_rgb.shape[1],
            'image_height': self.image_rgb.shape[0],
            'bboxes': self.bboxes
        }
        os.makedirs(os.path.dirname(self.annotation_file), exist_ok=True)
        with open(self.annotation_file, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"✓ Сохранено в {self.annotation_file}")
    
    def annotate_interactive(self):
        """Интерактивная разметка"""
        fig, ax = plt.subplots(figsize=(14, 10))
        ax.imshow(self.image_rgb)
        ax.set_title(f"Кликни и тащи для разметки печатей/меток. Файл: {Path(self.image_path).name}")
        
        # Отображаем существующие боксы
        for bbox in self.bboxes:
            x1, y1, x2, y2 = bbox['coords']
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                     linewidth=2, edgecolor='green', facecolor='none')
            ax.add_patch(rect)
        
        def on_select(eclick, erelease):
            x1, y1 = int(eclick.xdata), int(eclick.ydata)
            x2, y2 = int(erelease.xdata), int(erelease.ydata)
            
            # Нормализуем координаты
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            
            self.bboxes.append({
                'coords': [x1, y1, x2, y2],
                'width': x2 - x1,
                'height': y2 - y1
            })
            
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            fig.canvas.draw_idle()
            print(f"Разметка добавлена: ({x1}, {y1}, {x2}, {y2})")
        
        rect_selector = RectangleSelector(
            ax, on_select,
            useblit=True,
            button=[1],  # Left mouse button
            minspanx=5, minspany=5,
            spancoords='pixels',
            interactive=True
        )
        
        plt.show()
        return self.bboxes

print("✓ Класс ImageAnnotator готов")

## 5. Разметка изображений (интерактивно)

In [ ]:
# Показываем первые несколько изображений для разметки
images_to_annotate = sorted(Path(PAGES_FOLDER).glob("*.png"))[:10]  # Первые 10 для начала

for img_path in images_to_annotate:
    ann_file = os.path.join(ANNOTATIONS_FOLDER, f"{img_path.stem}.json")
    
    # Пропускаем если уже размечено
    if os.path.exists(ann_file):
        print(f"⏭️  {img_path.name} уже размечено")
        continue
    
    print(f"\n📍 Разметка {img_path.name}")
    print("Инструкция: кликни и тащи мышь для создания бокса вокруг печати/метки")
    print("После завершения закрой график (X кнопка)\n")
    
    annotator = ImageAnnotator(str(img_path), ann_file)
    annotator.annotate_interactive()
    annotator.save_annotations()
    print(f"✓ Разметка сохранена")

## 6. Проверка аннотаций

In [ ]:
# Загружаем статистику по разметке
annotation_files = list(Path(ANNOTATIONS_FOLDER).glob("*.json"))
stats = []

for ann_file in annotation_files:
    with open(ann_file, 'r') as f:
        data = json.load(f)
        stats.append({
            'image': Path(data['image']).name,
            'num_stamps': len(data['bboxes'])
        })

df_stats = pd.DataFrame(stats)
print("Статистика по разметке:")
print(df_stats)
print(f"\nВсего размечено изображений: {len(annotation_files)}")
print(f"Всего печатей/меток: {df_stats['num_stamps'].sum()}")

## 7. Подготовка данных для YOLO

In [ ]:
def create_yolo_dataset(annotations_folder, images_folder, output_folder, train_ratio=0.8):
    """
    Создает датасет в формате YOLO
    """
    # Создаем структуру папок
    for split in ['train', 'val']:
        os.makedirs(f"{output_folder}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_folder}/{split}/labels", exist_ok=True)
    
    # Получаем все аннотации
    annotation_files = sorted(Path(annotations_folder).glob("*.json"))
    
    # Разделяем на train/val
    split_idx = int(len(annotation_files) * train_ratio)
    train_files = annotation_files[:split_idx]
    val_files = annotation_files[split_idx:]
    
    for split, files in [("train", train_files), ("val", val_files)]:
        for ann_file in files:
            with open(ann_file, 'r') as f:
                data = json.load(f)
            
            img_path = data['image']
            img_name = Path(img_path).name
            
            # Копируем изображение
            import shutil
            shutil.copy(img_path, f"{output_folder}/{split}/images/{img_name}")
            
            # Создаем YOLO label файл
            width = data['image_width']
            height = data['image_height']
            
            label_content = ""
            for bbox in data['bboxes']:
                x1, y1, x2, y2 = bbox['coords']
                
                # Конвертируем в YOLO формат (center_x, center_y, width, height - нормализованные)
                center_x = (x1 + x2) / 2 / width
                center_y = (y1 + y2) / 2 / height
                norm_width = (x2 - x1) / width
                norm_height = (y2 - y1) / height
                
                label_content += f"0 {center_x:.6f} {center_y:.6f} {norm_width:.6f} {norm_height:.6f}\n"
            
            # Сохраняем label файл
            label_file = f"{output_folder}/{split}/labels/{Path(img_name).stem}.txt"
            with open(label_file, 'w') as f:
                f.write(label_content)
    
    print(f"✓ Train images: {len(train_files)}")
    print(f"✓ Val images: {len(val_files)}")
    print(f"✓ Датасет сохранен в {output_folder}")

# Создаем датасет
if len(list(Path(ANNOTATIONS_FOLDER).glob("*.json"))) > 0:
    create_yolo_dataset(ANNOTATIONS_FOLDER, PAGES_FOLDER, DATASET_FOLDER)
else:
    print("⚠️  Сначала размечь изображения!")

## 8. Обучение YOLOv8 модели

In [ ]:
from ultralytics import YOLO

# Создаем data.yaml для YOLO
yaml_content = f"""path: {os.path.abspath(DATASET_FOLDER)}
train: train/images
val: val/images

nc: 1  # Один класс - печать/метка
names: ['stamp']  # Названия классов
"""

with open(f"{DATASET_FOLDER}/data.yaml", 'w') as f:
    f.write(yaml_content)

print("✓ data.yaml создан")
print(f"Путь к датасету: {os.path.abspath(DATASET_FOLDER)}")

In [ ]:
# Загружаем YOLOv8 модель и обучаем
model = YOLO('yolov8n.pt')  # nano модель для быстрого обучения

# Обучаем
results = model.train(
    data=f"{DATASET_FOLDER}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    device=0,  # GPU если доступен
    verbose=True
)

print("✓ Обучение завершено!")

## 9. Тестирование на новом PDF

In [ ]:
def detect_stamps_in_pdf(pdf_path, model, confidence=0.5):
    """
    Детектирует печати в PDF и возвращает координаты
    """
    # Конвертируем PDF в изображения
    temp_folder = "temp_test"
    images = convert_pdf_to_images(pdf_path, temp_folder)
    
    results_data = {}
    
    for img_path in images:
        # Детектируем объекты
        results = model.predict(img_path, conf=confidence)
        
        detections = []
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = box.conf[0].item()
                detections.append({
                    'coords': [x1, y1, x2, y2],
                    'confidence': conf
                })
        
        results_data[img_path] = detections
    
    return results_data

print("✓ Функция detect_stamps_in_pdf готова")

In [ ]:
# Пример использования - детектируем на первом PDF
model = YOLO("runs/detect/train/weights/best.pt")

test_pdf = list(Path(INPUT_FOLDER).glob("*.pdf"))[0]
print(f"Тестируем на: {test_pdf}")

detections = detect_stamps_in_pdf(str(test_pdf), model, confidence=0.4)

for img_path, stamps in detections.items():
    print(f"\n{Path(img_path).name}: найдено {len(stamps)} печатей/меток")
    for i, stamp in enumerate(stamps):
        conf = stamp['confidence']
        print(f"  #{i+1}: confidence={conf:.2f}")

## 10. Визуализация результатов

In [ ]:
def visualize_detections(img_path, detections, figsize=(14, 10)):
    """
    Отображает изображение с детектированными печатями
    """
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image_rgb)
    
    for detection in detections:
        x1, y1, x2, y2 = detection['coords']
        conf = detection.get('confidence', 0)
        
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor='blue', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{conf:.2f}", color='blue', fontsize=10, weight='bold')
    
    ax.set_title(f"Детектированные печати/метки ({len(detections)} найдено)")
    plt.tight_layout()
    plt.show()

# Пример
for img_path, stamps in list(detections.items())[:2]:
    if stamps:
        visualize_detections(img_path, stamps)